## 1. Load Dataset and Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [2]:
df = pd.read_csv("fifa21_cleaned.csv")

In [3]:
df.head(10)

,photoUrl,LongName,playerUrl,Nationality,Positions,Name,Age,OVA,POT,ID,...,pos_GK,pos_LB,pos_LM,pos_LW,pos_LWB,pos_RB,pos_RM,pos_RW,pos_RWB,pos_ST
0,https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/2...,Argentina,RW ST CF,L. Messi,33,93,93,158023,...,0,0,0,0,0,0,0,1,0,1
1,https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-s...,Portugal,ST LW,Cristiano Ronaldo,35,92,92,20801,...,0,0,0,1,0,0,0,0,0,1
2,https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,200389,...,1,0,0,0,0,0,0,0,0,0
3,https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyn...,Belgium,CAM CM,K. De Bruyne,29,91,91,192985,...,0,0,0,0,0,0,0,0,0,0
4,https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silv...,Brazil,LW CAM,Neymar Jr,28,91,91,190871,...,0,0,0,1,0,0,0,0,0,0
5,https://cdn.sofifa.com/players/188/545/21_60.png,Robert Lewandowski,http://sofifa.com/player/188545/robert-lewando...,Poland,ST,R. Lewandowski,31,91,91,188545,...,0,0,0,0,0,0,0,0,0,1
6,https://cdn.sofifa.com/players/231/747/21_60.png,Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/...,France,ST LW RW,K. Mbappé,21,90,95,231747,...,0,0,0,1,0,0,0,1,0,1
7,https://cdn.sofifa.com/players/212/831/21_60.png,Alisson Ramses Becker,http://sofifa.com/player/212831/alisson-ramses...,Brazil,GK,Alisson,27,90,91,212831,...,1,0,0,0,0,0,0,0,0,0
8,https://cdn.sofifa.com/players/209/331/21_60.png,Mohamed Salah,http://sofifa.com/player/209331/mohamed-salah/...,Egypt,RW,M. Salah,28,90,90,209331,...,0,0,0,0,0,0,0,1,0,0
9,https://cdn.sofifa.com/players/208/722/21_60.png,Sadio Mané,http://sofifa.com/player/208722/sadio-mane/210...,Senegal,LW,S. Mané,28,90,90,208722,...,0,0,0,1,0,0,0,0,0,0


In [4]:
df[['Wage',"Name"]]

,Wage,Name
0,560000.0,L. Messi
1,220000.0,Cristiano Ronaldo
2,125000.0,J. Oblak
3,370000.0,K. De Bruyne
4,270000.0,Neymar Jr
...,...,...
18973,1000.0,Zhang Mengxuan
18974,500.0,V. Da Silva
18975,1000.0,Xia Ao
18976,500.0,B. Hough


In [5]:
df.columns

Index(['photoUrl', 'LongName', 'playerUrl', 'Nationality', 'Positions', 'Name',
       'Age', 'OVA', 'POT', 'ID', 'Height', 'Weight', 'foot', 'BOV', 'BP',
       'Growth', 'Joined', 'Loan Date End', 'Value', 'Wage', 'Release Clause',
       'Attacking', 'Crossing', 'Finishing', 'Heading Accuracy',
       'Short Passing', 'Volleys', 'Skill', 'Dribbling', 'Curve',
       'FK Accuracy', 'Long Passing', 'Ball Control', 'Movement',
       'Acceleration', 'Sprint Speed', 'Agility', 'Reactions', 'Balance',
       'Power', 'Shot Power', 'Jumping', 'Stamina', 'Strength', 'Long Shots',
       'Mentality', 'Aggression', 'Interceptions', 'Positioning', 'Vision',
       'Penalties', 'Composure', 'Defending', 'Marking', 'Standing Tackle',
       'Sliding Tackle', 'Goalkeeping', 'GK Diving', 'GK Handling',
       'GK Kicking', 'GK Positioning', 'GK Reflexes', 'Total Stats',
       'Base Stats', 'W/F', 'SM', 'A/W', 'D/W', 'IR', 'PAC', 'SHO', 'PAS',
       'DRI', 'DEF', 'PHY', 'Hits', 'Team', 'Contract

In [6]:
# column types
print(df.dtypes.value_counts())
df.select_dtypes(include=['float64']).nunique()

int64      74
str        14
float64     7
Name: count, dtype: int64


Height              21
Weight              56
Value              214
Wage               141
Release Clause    1216
Contract_start      20
Contract_end         9
dtype: int64

## 2. Define the problem

Between Wage and Value, I choose Wage to be predicted.

## 3. Feature Selection & Leakage Check

To check if a value is derivated from wage or not.

In [7]:
ratio = df['Value'] / df['Wage']
ratio.describe()

count    18741.000000
mean       460.747827
std       1454.554598
min          0.000000
25%        138.461538
50%        231.250000
75%        375.000000
max      51000.000000
dtype: float64

Result : Value is not derived from Wage, the ratio goes from 0 to 51000 so its not a fixed formula. Value can be used, no leakage.

In [8]:
df['Value'].isnull().sum(), df['Wage'].isnull().sum(), (df['Wage'] == 0).sum()

(np.int64(0), np.int64(0), np.int64(237))

Result: every row where Wage == 0 also has Value == 0 (237 rows, exact match). So the ratio for those was 0/0 = NaN, not infinity, and describe() silently drops NaN from its count. 18978 - 237 = 18741, matches what describe() showed earlier.

Side note: 11 more rows have Value == 0 but Wage is not 0 (248 - 237). Not a problem, just means the two zero-sets aren't identical, worth knowing.

### Next is POT and OVA

In [9]:
ratio = df['OVA'] / df['POT']
ratio.describe()

count    18978.000000
mean         0.925382
std          0.077098
min          0.670886
25%          0.870683
50%          0.945946
75%          1.000000
max          1.000000
dtype: float64

OVA / POT never goes above 1.0, which makes sense because your potential can't be lower than
what you already are. 75% of players are sitting at exactly 1.0 so they've already peaked, and
the lowest is 0.67 which would be a young player with a lot of growing left.

Spread is real (0.67 to 1.0) so these aren't the same column twice, but they are related by
design. Not a leakage problem, just something to remember later.

### Next is POT and Wage

In [10]:
ratio = df['Wage'] / df['POT']
ratio.describe()

count    18978.000000
mean       114.243138
std        231.366262
min          0.000000
25%         14.285714
50%         41.666667
75%        106.666667
max       6021.505376
dtype: float64

Mean 114 but std 231, so the spread is bigger than the average itself, and it goes from 0 to
6021. If POT was secretly Wage under another name this ratio would be nearly the same number
every row. It isn't, so POT is fine to use.

### Next is Release_clause and Wage

In [11]:
ratio = df['Release Clause'] / df['Wage']
ratio.describe()

count     18741.000000
mean        894.009682
std        3286.890070
min           0.000000
25%         216.000000
50%         378.000000
75%         650.000000
max      111000.000000
dtype: float64

Same story, mean 894 and std 3286, max 111000. Checked the top ratios and they're all players
with Wage exactly 500, which looks like a floor value rather than a real salary. Dividing by
something that small is what blows the ratio up, it's not a sign of a formula. Release Clause is
fine to use.

### Checking foot, Height and Weight

In [12]:
df.groupby('foot')['Wage'].mean()

foot
Left     9516.277018
Right    8570.683329
Name: Wage, dtype: float64

In [13]:
df.groupby('foot')['Wage'].median()

foot
Left     3000.0
Right    3000.0
Name: Wage, dtype: float64

In [14]:
df.groupby('BP').apply(lambda g: g['Height'].corr(g['Wage']))

BP
CAM    0.078773
CB     0.089026
CDM    0.091768
CF     0.237025
CM     0.039328
GK     0.120709
LB     0.004796
LM     0.062998
LW    -0.095291
LWB   -0.041293
RB     0.028441
RM     0.054392
RW    -0.037254
RWB    0.125804
ST     0.047591
dtype: float64

In [15]:
print(df['Height'].corr(df['Wage']))
print(df['Weight'].corr(df['Wage']))

0.022102185482022595
0.06524778413297246


Height and Weight both come out near zero (0.02 and 0.07), and even goalkeepers only hit
0.12, so the idea that height matters for keepers doesn't hold. Not using either of them.

foot is the same story. The mean looked like an 11% gap but the median is exactly 3000 for both,
so that gap was just a few high earners on the left-footed side. Not using it.

## 3b. Feature Engineering

Making new columns out of the ones we already have. Everything here is row-wise, meaning each
row is calculated only from itself and doesn't look at any other row, so it's safe to do before
the train/test split. Anything that needs a statistic across rows (like scaling, which needs a
mean) has to wait until after.

In [16]:
df['contract_length'] = df['Contract_end'] - df['Contract_start']
df['years_at_club'] = 2021 - df['Contract_start']

print(df[['contract_length','years_at_club']].describe())
print()
print(df.groupby('Contract_type')['Wage'].median())
print()
print('contract_length corr with Wage:', df['contract_length'].corr(df['Wage']))

       contract_length  years_at_club
count     17728.000000   17728.000000
mean          3.737816       2.673567
std           2.291615       2.091385
min           0.000000       1.000000
25%           2.000000       1.000000
50%           3.000000       2.000000
75%           5.000000       3.000000
max          23.000000      23.000000

Contract_type
Contract    3000.0
Free           0.0
On Loan     3000.0
Name: Wage, dtype: float64

contract_length corr with Wage: 0.24831040053746756


### Contract length

contract_length correlates 0.248 with Wage, which is about 10x anything else we've checked so far.
Weak on paper but it's the best feature we've found, keeping it.

### Handling the missing contract years

1250 rows have no Contract_start or Contract_end. Those are the On Loan and Free rows from
cleaning, where the parser set them to None on purpose because a loan or a free agent doesn't
have contract years.

Checked what those two groups actually look like:

In [17]:
df.groupby('Contract_type')['Wage'].agg(['count', 'median', 'mean', 'min', 'max'])

,count,median,mean,min,max
Contract_type,,,,,
Contract,17728,3000.0,8941.031137,500.0,560000.0
Free,237,0.0,0.000000,0.0,0.0
On Loan,1013,3000.0,8326.900296,500.0,130000.0


Free agents are all exactly 0. Every single one of the 237. That makes sense since they
have no club, but it also means they're not really the same question we're asking. We want to
predict what a club would pay a player, and these players don't have a club. Leaving them in
would teach the model that some players with decent skills earn nothing, which is about
employment status not about skill. Dropping them.

On Loan is different. Same median as contracted players (3000), same minimum, so they're real
players earning real wages. They just don't have contract years recorded because their loan end
date lives in Loan Date End instead. Keeping them and filling the gap.

In [18]:
df = df[df['Contract_type'] != 'Free'].copy()

df['on_loan'] = (df['Contract_type'] == 'On Loan').astype(int)
df['contract_length'] = df['contract_length'].fillna(0)
df['years_at_club'] = df['years_at_club'].fillna(0)

print(df.shape)
print('nulls left:', df['contract_length'].isnull().sum(), df['years_at_club'].isnull().sum())
df.groupby('on_loan')['contract_length'].describe()[['count', 'mean', 'min', 'max']]

(18741, 98)
nulls left: 0 0


,count,mean,min,max
on_loan,,,,
0,17728.0,3.737816,0.0,23.0
1,1013.0,0.000000,0.0,0.0


The on_loan flag matters here. Without it, filling with 0 tells the model these players
have a zero year contract, which isn't true. And we already found a real contract_length == 0
group (361 players, median wage 750), so we'd be mixing two completely different situations into
the same number.

With the flag the model can tell them apart. contract_length=0 with on_loan=0 means a real
same-year contract. contract_length=0 with on_loan=1 means no contract data because they're on
loan. The 0 stops being a claim about reality and the flag carries the actual meaning.

General rule worth remembering: when you fill missing values, add a flag for which rows got
filled. The fact that something was missing is usually information by itself.

### Leakage check against Wage with OVA

In [19]:
print(df['OVA'].corr(df['Wage']))
check_OVA = df['Wage'] / df['OVA']
check_OVA.describe()


0.5913198502678958


count    18741.000000
mean       120.349815
std        236.519500
min          6.024096
25%         16.393443
50%         45.454545
75%        114.285714
max       6021.505376
dtype: float64

As we can see, OVA can be used against Wage prediction because there's no pattern that could lead into leakage

### BOV and Growth check against Wage

In [20]:
print(df['BOV'].corr(df['Wage']))
check_BOV = df['Wage'] / df['BOV']
print(check_BOV.describe())

print(df['Growth'].corr(df['Wage']))
growth = df['POT'] - df['OVA']
print('Growth is exactly POT - OVA:', (growth == df['Growth']).all())
print('rows where BOV == OVA:', (df['BOV'] == df['OVA']).mean())

0.5929103234392572
count    18741.000000
mean       119.024965
std        235.065656
min          6.024096
25%         15.873016
50%         44.776119
75%        112.676056
max       6021.505376
dtype: float64
-0.21037074090244043
Growth is exactly POT - OVA: True
rows where BOV == OVA: 0.4446400939117443


Growth is exactly POT - OVA, and BOV is the same as OVA in 44% of rows. Both are redundant, so we keep OVA.

Drop : Growth and BOV

### Dropping BOV and Growth

In [21]:
# Drop BOV and Growth columns
df = df.drop(['BOV', 'Growth'], axis=1)

### Total Stats and Base Stats

Same thing as the other 7 sums. Base Stats is exactly PAC + SHO + PAS + DRI + DEF + PHY, checked
it and it's True for every row. Total Stats is the same idea over more columns. The detector we
wrote earlier missed these two because it only looks at the columns immediately after, and PAC
through PHY sit at the end of the file instead. Caught them by noticing Total Stats maxes at
2316 which is way past any 0-99 rating.

In [22]:
base = ['PAC','SHO','PAS','DRI','DEF','PHY']
(df['Base Stats'] == df[base].sum(axis=1)).all()

np.True_

### Team

681 clubs. Median wage goes from 137500 at Real Madrid down to 500 at the smaller clubs, so this
is probably the strongest signal in the whole dataset, maybe stronger than OVA. Which club you
play for basically is your wage.

Problem is one hot encoding turns it into 681 columns, mostly zeros. Checked and no team has
fewer than 5 players so it's not as bad as Nationality would have been, but it's still a lot of
width and makes it easy for the model to just memorise clubs instead of learning a pattern.

Dropping it for now and coming back in the iterate phase, so we can actually measure how much it
adds instead of guessing. If we do bring it back, better idea is grouping clubs into wage tiers
rather than 681 separate columns, but that uses Wage to build a feature so it has to happen after
the split.

### Dropping everything we decided against

In [23]:
drop_cols = [
    # identifiers, they tell you who not why they're paid
    'photoUrl', 'playerUrl', 'LongName', 'Name', 'ID',

    # checked, no relationship with Wage
    'foot', 'Height', 'Weight',

    # too many categories
    'Nationality', 'Team',

    # redundant (BOV and Growth already dropped above)
    'Positions',      # already have pos_ columns
    'Joined',         # same info as Contract_start
    'Contract_type',  # on_loan covers it
    'Loan Date End',  # date, and on_loan covers it

    # raw years, contract_length and years_at_club already cover these
    'Contract_start', 'Contract_end',

    # sums of columns we're keeping
    'Attacking', 'Skill', 'Movement', 'Power', 'Mentality',
    'Defending', 'Goalkeeping', 'Total Stats', 'Base Stats',
]

df = df.drop(columns=drop_cols)
df.shape

(18741, 71)

### One hot encoding

BP, A/W and D/W are still text and models can't read text. One hot turns each category into its
own 0/1 column, so BP becomes BP_CAM, BP_CB, BP_GK and so on.

This is row-wise (is this row's BP equal to GK, yes or no) so it doesn't look at any other row,
which means it's safe to do before the split.

In [24]:
df = pd.get_dummies(df, columns=['BP', 'A/W', 'D/W'], dtype=int)

print(df.shape)
print([c for c in df.columns if c.startswith(('BP_', 'A/W_', 'D/W_'))][:8])

(18741, 89)
['BP_CAM', 'BP_CB', 'BP_CDM', 'BP_CF', 'BP_CM', 'BP_GK', 'BP_LB', 'BP_LM']


In [25]:
# nothing should be text anymore
df.select_dtypes('object').columns.tolist()

[]

### Where section 3 ended up

95 columns at the start, 3 new ones, 27 dropped, and one hot turned BP, A/W and D/W into 21. So 89 in the end (88 features + Wage), all numeric, no nulls, ready to split.

Strongest features : OVA (0.591) and contract_length (0.248).

## 4. Train/Test Split

In [26]:
X = df.drop('Wage', axis=1)
y = df['Wage']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((14992, 88), (3749, 88), (14992,), (3749,))

## 5. Baseline

In [27]:
mean_guess = y_train.mean()
median_guess = y_train.median()

for name, guess in [('mean', mean_guess), ('median', median_guess)]:
    pred = np.full(len(y_test), guess)          # the constant, repeated 3749 times
    mae = mean_absolute_error(y_test, pred)
    rmse = root_mean_squared_error(y_test, pred)
    print(f'{name:7} guess={guess:9.0f}   MAE={mae:9.0f}   RMSE={rmse:9.0f}')

mean    guess=     8872   MAE=     9867   RMSE=    21203
median  guess=     3000   MAE=     7754   RMSE=    22049


In [28]:
pred_baseline = np.full(len(y_test), median_guess)

For MAE the Median wins, because half the players earn 3000 or less so guessing 3000 is close for a lot of them.

For RMSE the Mean wins, because RMSE squares the errors, so guessing 3000 for someone earning 560000 gets punished really hard.

## 6. Training Model

In [29]:
# Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE :', mean_absolute_error(y_test, pred))
print('RMSE:', root_mean_squared_error(y_test, pred))

MAE : 4435.322148768088
RMSE: 10865.548868155527


In [30]:
# Random Forest
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE :', mean_absolute_error(y_test, pred))
print('RMSE:', root_mean_squared_error(y_test, pred))

MAE : 3986.411309682582
RMSE: 10469.934037592297


In [31]:
# Gradient Boosting
model = GradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE :', mean_absolute_error(y_test, pred))
print('RMSE:', root_mean_squared_error(y_test, pred))

MAE : 3984.516171501596
RMSE: 10307.61464250974


In [32]:
# SVR
model = SVR()
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE :', mean_absolute_error(y_test, pred))
print('RMSE:', root_mean_squared_error(y_test, pred))

MAE : 7364.487630253149
RMSE: 21547.50803077726


In [33]:
# XGBoost
model = XGBRegressor()
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE :', mean_absolute_error(y_test, pred))
print('RMSE:', root_mean_squared_error(y_test, pred))

MAE : 4102.144380885272
RMSE: 11208.911850719358


In [34]:
# LightGBM
model = LGBMRegressor(verbose=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print('MAE :', mean_absolute_error(y_test, pred))
print('RMSE:', root_mean_squared_error(y_test, pred))

MAE : 3973.604160020143
RMSE: 10438.674788081044


### Save the best model

In [35]:
model_lightgbm = LGBMRegressor(verbose=-1)
model_lightgbm.fit(X_train, y_train)
pred_lightgbm = model_lightgbm.predict(X_test)

print('MAE :', mean_absolute_error(y_test, pred_lightgbm))
print('RMSE:', root_mean_squared_error(y_test, pred_lightgbm))

MAE : 3973.604160020143
RMSE: 10438.674788081044


## 7. Evaluate

### Comparing models against the baseline

In [36]:
print("Baseline")
print(f"MAE : {mean_absolute_error(y_test, pred_baseline):.0f}")
print(f"RMSE: {root_mean_squared_error(y_test, pred_baseline):.0f}")

print("Best Model (LightGBM)")
print(f"MAE : {mean_absolute_error(y_test, pred_lightgbm):.0f}")
print(f"RMSE: {root_mean_squared_error(y_test, pred_lightgbm):.0f}")

Baseline
MAE : 7754
RMSE: 22049
Best Model (LightGBM)
MAE : 3974
RMSE: 10439


### Overfitting check

In [37]:
train_pred = model_lightgbm.predict(X_train)
test_pred  = model_lightgbm.predict(X_test)

print('train MAE:', mean_absolute_error(y_train, train_pred))
print('test  MAE:', mean_absolute_error(y_test, test_pred))

train MAE: 2646.4161842947096
test  MAE: 3973.604160020143


Train MAE 2646 vs test MAE 3974, so the test is around 50% worse. Mild overfitting, normal for boosting on default settings.

### Feature Importance check

In [38]:
importance = pd.Series(model_lightgbm.feature_importances_, index=X_train.columns)
importance.sort_values(ascending=False).head(15)

Hits               250
Value              195
Release Clause     161
Age                128
OVA                126
Composure           93
POT                 76
Reactions           72
contract_length     67
Ball Control        66
Short Passing       65
Strength            59
Volleys             58
Penalties           57
Crossing            54
dtype: int32

In [39]:
# share of the total gain, so it adds up to 1
importance = pd.Series(
    model_lightgbm.booster_.feature_importance(importance_type='gain'),
    index=X_train.columns
)
(importance / importance.sum()).sort_values(ascending=False).head(15)

Value               0.420355
OVA                 0.363532
Hits                0.032110
Release Clause      0.024886
Ball Control        0.018619
Age                 0.012957
IR                  0.011462
Composure           0.011275
contract_length     0.005410
Reactions           0.004805
Sliding Tackle      0.004801
Penalties           0.003882
Stamina             0.003595
POT                 0.003489
Heading Accuracy    0.003451
dtype: float64

Value takes 42% of the total gain and OVA 36%, so the model mostly leans on those two.

Suggestion : Drop Value or OVA so the model uses the other information. (Tested in section 10.)

## 8. Diagnose Errors

In [40]:
result = pd.DataFrame({'actual' : y_test, 'predicted' : model_lightgbm.predict(X_test)})
result['error'] = result['predicted'] - result['actual']
result['abs_error'] = result['error'].abs()
result.sort_values('abs_error', ascending=False).head(10)

,actual,predicted,error,abs_error
0,560000.0,263163.660992,-296836.339008,296836.339008
59,58000.0,232847.050710,174847.050710,174847.050710
57,93000.0,212268.515459,119268.515459,119268.515459
113,20000.0,119227.033571,99227.033571,99227.033571
254,125000.0,28592.064393,-96407.935607,96407.935607
3,370000.0,282239.199962,-87760.800038,87760.800038
149,160000.0,72240.182152,-87759.817848,87759.817848
1113,125000.0,46384.239906,-78615.760094,78615.760094
368,12000.0,85356.765048,73356.765048,73356.765048
102,47000.0,120292.270401,73292.270401,73292.270401


In [41]:
w = int(len(result) * 0.01)
result.nlargest(w, 'abs_error')['abs_error'].sum() / result['abs_error'].sum()

np.float64(0.17175017610793708)

In [42]:
# error by wage band
bands = [0, 3000, 10000, 50000, 1000000]
result.groupby(pd.cut(result['actual'], bands), observed=True).agg(
    n=('error', 'size'),
    mean_error=('error', 'mean'),
    mae=('abs_error', 'mean'),
)

,n,mean_error,mae
actual,,,
"(0, 3000]",2114,1172.408132,1506.962298
"(3000, 10000]",858,660.414824,2873.141044
"(10000, 50000]",658,-2185.982814,7809.229493
"(50000, 1000000]",119,-18188.030089,34518.451061


By the result, we can conclude that the problem comes from group of superstars. It leads to the uneven spread of error. So, the 1% of the test set which is 37 players, carries almost 17.2% of the total error.

That implies to the 3974 MAE is not the model being slightly wrong everywhere, its mostly a small group with very big errors.

Worth to be mentioned that the wage band table gives insights.

From the wage table, MAE grows with wage increasing. 1506 for players under 3000, and up to 34518 MAE for top band. It means the model gets worse 20+ times when the wage increases.

Beside that, we also see that the direction of the Mean Error. For the bottom of wage band, it gives positive (+1172) which means the model guesses too high for lower band. For the top wage band, it gives a big negative (-18188) which means it guesses too low for higher band.

The important part is that every band leans to one side consistently. If the model was
just randomly wrong, the plus and the minus would cancel each other and mean_error would
sit close to 0. It doesnt. So this is a systematic bias, not noise. The model pushes
everyone to the middle, lifting the cheap players up and flattening the expensive ones down.

Why this happens : the loss counts the error in euro. Missing by 5000 on a player who
earns 2000 and missing by 5000 on a player who earns 300000 look exactly the same to the
model, even though the first one is way worse. And since most of the players are cheap,
the model gets a better total score by fitting the crowd and giving up on the 119 top
earners.

So the wall is not the algorithm. We already tried 6 of them and they all stopped around
4000. The problem is the target itself being too skewed.

Suggestion : train on log(Wage) instead of Wage so the model stops measuring in euro,
then re-run this same table and check if the sign flip is gone. That is phase 9.

## 9. Experiment

### Training on log(Wage) instead of Wage

In [43]:
y_train_log = np.log1p(y_train)

model_log = LGBMRegressor(verbose=-1)
model_log.fit(X_train, y_train_log)

pred_log = np.expm1(model_log.predict(X_test))

print('MAE :', mean_absolute_error(y_test, pred_log))
print('RMSE:', root_mean_squared_error(y_test, pred_log))

MAE : 3920.9207089283427
RMSE: 10739.184190163838


In [44]:
result_log = pd.DataFrame({'actual': y_test, 'predicted': pred_log})
result_log['error'] = result_log['predicted'] - result_log['actual']
result_log['abs_error'] = result_log['error'].abs()

w = int(len(result_log) * 0.01)
print('top 1% share of error:', result_log.nlargest(w, 'abs_error')['abs_error'].sum() / result_log['abs_error'].sum())

result_log.groupby(pd.cut(result_log['actual'], bands), observed=True).agg(
    n=('error', 'size'),
    mean_error=('error', 'mean'),
    mae=('abs_error', 'mean'),
)

top 1% share of error: 0.17592127626559143


,n,mean_error,mae
actual,,,
"(0, 3000]",2114,514.983607,993.804587
"(3000, 10000]",858,-660.237461,2536.483185
"(10000, 50000]",658,-6277.169290,9031.360719
"(50000, 1000000]",119,-32951.482250,37644.461476


In [45]:
result['pct_error'] = result['abs_error'] / result['actual']
result_log['pct_error'] = result_log['abs_error'] / result_log['actual']

for name, r in [('euro model', result), ('log model', result_log)]:
    print(name)
    print(r.groupby(pd.cut(r['actual'], bands), observed=True)['pct_error'].mean())
    print()

euro model
actual
(0, 3000]           2.091664
(3000, 10000]       0.482460
(10000, 50000]      0.374426
(50000, 1000000]    0.385550
Name: pct_error, dtype: float64

log model
actual
(0, 3000]           1.248371
(3000, 10000]       0.427029
(10000, 50000]      0.418279
(50000, 1000000]    0.410612
Name: pct_error, dtype: float64



With log, MAE went a bit lower (3974 to 3921) but RMSE went higher (10439 to 10739). The 0-3000 band got much better (209% to 125%), but the top two bands got a bit worse.

Update after phase 9 : this diagnosis was made using euro error, and it turned out to be misleading.
When we measure the error as a percentage of the actual wage, the ranking flips. The bottom band is
the worst (209% error) and the top band is actually the best (39%). The top only looked bad in euro
because their wages are big, so any miss is big in euro.

So the model was not giving up on the superstars. It was being very inaccurate on the cheap players,
because a 1000 euro miss is small to the loss even when it is double the player's wage. The euro
table hid that completely.

## 10. Experiment #2

### Dropping Value from the prediction

In [46]:
# same rows as the original split, just without the Value column
X_train_no_value = X_train.drop(columns=['Value'])
X_test_no_value = X_test.drop(columns=['Value'])

In [47]:
X_train_no_value.shape, X_test_no_value.shape

((14992, 87), (3749, 87))

In [48]:
model_no_value = LGBMRegressor(verbose=-1)
model_no_value.fit(X_train_no_value, y_train)

pred_without_value = model_no_value.predict(X_test_no_value)

print('MAE :', mean_absolute_error(y_test, pred_without_value))
print('RMSE:', root_mean_squared_error(y_test, pred_without_value))

MAE : 3958.1324446198714
RMSE: 10374.932015633754


In [49]:
model_no_value_log = LGBMRegressor(verbose=-1)
model_no_value_log.fit(X_train_no_value, y_train_log)

log_pred_without_value = np.expm1(model_no_value_log.predict(X_test_no_value))

print('MAE :', mean_absolute_error(y_test, log_pred_without_value))
print('RMSE:', root_mean_squared_error(y_test, log_pred_without_value))

MAE : 4029.8179730692395
RMSE: 11327.48046250793


In [50]:
# feature importance without value
importance = pd.Series(
    model_no_value.booster_.feature_importance(importance_type='gain'),
    index=X_train_no_value.columns
)
importance.sort_values(ascending=False).head(15)

OVA                 1.873459e+13
Release Clause      2.983029e+12
Hits                1.089130e+12
Ball Control        4.086551e+11
Age                 3.974609e+11
IR                  3.622386e+11
Composure           2.580549e+11
Reactions           2.067790e+11
Penalties           1.434764e+11
POT                 1.352770e+11
Jumping             1.306396e+11
Stamina             1.250414e+11
Heading Accuracy    1.200575e+11
Interceptions       1.095353e+11
Aggression          1.030681e+11
dtype: float64

### Dropping Release_clause and Value

In [51]:
# same rows again, this time without Value and Release Clause
X_train_no_value_rc = X_train.drop(columns=['Value', 'Release Clause'])
X_test_no_value_rc = X_test.drop(columns=['Value', 'Release Clause'])

In [52]:
model_no_value_rc = LGBMRegressor(verbose=-1)
model_no_value_rc.fit(X_train_no_value_rc, y_train)

pred_without_value_and_release = model_no_value_rc.predict(X_test_no_value_rc)

print('MAE :', mean_absolute_error(y_test, pred_without_value_and_release))
print('RMSE:', root_mean_squared_error(y_test, pred_without_value_and_release))

MAE : 4051.074550757725
RMSE: 10700.392625680923


In [53]:
model_no_value_rc_log = LGBMRegressor(verbose=-1)
model_no_value_rc_log.fit(X_train_no_value_rc, y_train_log)

log_pred_without_value_rc = np.expm1(model_no_value_rc_log.predict(X_test_no_value_rc))

print('MAE :', mean_absolute_error(y_test, log_pred_without_value_rc))
print('RMSE:', root_mean_squared_error(y_test, log_pred_without_value_rc))

MAE : 3961.7322730397736
RMSE: 10696.125651496808


In [54]:
# feature importance without value and release clause
importance = pd.Series(
    model_no_value_rc.booster_.feature_importance(importance_type='gain'),
    index=X_train_no_value_rc.columns
)
importance.sort_values(ascending=False).head(15)

OVA                2.131511e+13
Hits               1.283229e+12
Ball Control       4.947535e+11
Composure          3.250072e+11
Age                3.044843e+11
Reactions          2.231240e+11
Stamina            2.044232e+11
POT                1.841036e+11
Penalties          1.703251e+11
Long Passing       1.276738e+11
GK Handling        1.244347e+11
contract_length    1.233981e+11
IR                 1.177226e+11
Jumping            1.168103e+11
Crossing           1.033544e+11
dtype: float64

Removing only Value makes MAE 16 euro lower (3974 to 3958), removing Value and Release Clause makes it higher (4051). One split isnt enough to trust 16 euro, so we test it with K-fold.

In [55]:
folds = KFold(n_splits=5, shuffle=True, random_state=42)

with_value = -cross_val_score(LGBMRegressor(verbose=-1), X_train, y_train,
                              cv=folds, scoring='neg_mean_absolute_error')
without_value = -cross_val_score(LGBMRegressor(verbose=-1), X_train_no_value, y_train,
                                 cv=folds, scoring='neg_mean_absolute_error')

print('with value   :', with_value.round(0))
print('without value:', without_value.round(0))
print('difference   :', (without_value - with_value).round(0))

with value   : [3864. 3874. 3791. 3796. 3932.]
without value: [3821. 3963. 3839. 3810. 3987.]
difference   : [-44.  89.  49.  14.  55.]


Conclusion : the 16 euro was luck. With K-fold the model without Value is worse in 4 of 5 rounds, so we keep Value.

## 11. Experiment #3

We're bringing **team** back.

The plan is to extract median wage from each club. From 600+ columns to one column that is club_wage.

Why dont we use one hot encoding? because the model is just gonna memorize it than learning from it if we use 600+ columns to insert the team as a predicting value.


### Bringing Team back

Load Team and Wage again from the csv, and check the rows still line up with X_train and X_test before using Team.

In [56]:
raw = pd.read_csv('fifa21_cleaned.csv', usecols=['Team', 'Wage'])
print('train rows line up:', (raw.loc[X_train.index, 'Wage'] == y_train).all())
print('test rows line up :', (raw.loc[X_test.index, 'Wage'] == y_test).all())

team_train = raw.loc[X_train.index, 'Team']
team_test = raw.loc[X_test.index, 'Team']

print('nulls:', team_train.isnull().sum(), team_test.isnull().sum())
print('smallest club in train:', team_train.value_counts().min(), 'players')
print('test clubs never seen in train:', (~team_test.isin(team_train)).sum())

train rows line up: True
test rows line up : True
nulls: 0 0
smallest club in train: 13 players
test clubs never seen in train: 0


### The club wage function

A function for the median wage of each club. The medians only come from training rows, so the test set never sees its own wages.

In [57]:
def club_wage(fit_teams, fit_wages, apply_teams):
    medians = fit_wages.groupby(fit_teams).median()
    return apply_teams.map(medians).fillna(fit_wages.median())


X_train_team = X_train.assign(club_wage=club_wage(team_train, y_train, team_train))
X_test_team = X_test.assign(club_wage=club_wage(team_train, y_train, team_test))

X_train_team.shape, X_test_team.shape

((14992, 89), (3749, 89))

In [58]:
model_team = LGBMRegressor(verbose=-1)
model_team.fit(X_train_team, y_train)
pred_team = model_team.predict(X_test_team)

print('train MAE:', mean_absolute_error(y_train, model_team.predict(X_train_team)))
print('test  MAE:', mean_absolute_error(y_test, pred_team))
print('RMSE     :', root_mean_squared_error(y_test, pred_team))

train MAE: 1000.5916206987523
test  MAE: 1532.5145041123612
RMSE     : 5912.019752056152


### Percentage error by wage band, before and after Team

In [59]:
result_team = pd.DataFrame({'actual': y_test, 'predicted': pred_team})
result_team['pct_error'] = (result_team['predicted'] - result_team['actual']).abs() / result_team['actual']

pd.DataFrame({
    'without team': result.groupby(pd.cut(result['actual'], bands), observed=True)['pct_error'].mean(),
    'with team': result_team.groupby(pd.cut(result_team['actual'], bands), observed=True)['pct_error'].mean(),
})

,without team,with team
actual,,
"(0, 3000]",2.091664,0.329357
"(3000, 10000]",0.482460,0.192211
"(10000, 50000]",0.374426,0.168444
"(50000, 1000000]",0.385550,0.129896


In [60]:
# share of the total gain, so it adds up to 1
importance = pd.Series(
    model_team.booster_.feature_importance(importance_type='gain'),
    index=X_train_team.columns
)
(importance / importance.sum()).sort_values(ascending=False).head(10)

OVA             0.475646
club_wage       0.288747
Value           0.189197
Age             0.011889
Composure       0.006875
Ball Control    0.004290
Positioning     0.002521
Finishing       0.002428
Penalties       0.001590
Volleys         0.001274
dtype: float64

### Checking it with K-fold

In [61]:
with_team = []

for fit_pos, check_pos in folds.split(X_train):
    X_fit, X_check = X_train.iloc[fit_pos], X_train.iloc[check_pos]
    y_fit, y_check = y_train.iloc[fit_pos], y_train.iloc[check_pos]
    t_fit, t_check = team_train.iloc[fit_pos], team_train.iloc[check_pos]

    X_fit = X_fit.assign(club_wage=club_wage(t_fit, y_fit, t_fit))
    X_check = X_check.assign(club_wage=club_wage(t_fit, y_fit, t_check))

    m = LGBMRegressor(verbose=-1).fit(X_fit, y_fit)
    with_team.append(mean_absolute_error(y_check, m.predict(X_check)))

with_team = np.array(with_team)

print('without team:', with_value.round(0), 'mean:', with_value.mean().round(0))
print('with team   :', with_team.round(0), 'mean:', with_team.mean().round(0))
print('difference  :', (with_team - with_value).round(0))

without team: [3864. 3874. 3791. 3796. 3932.] mean: 3851.0
with team   : [1477. 1654. 1558. 1597. 1693.] mean: 1596.0
difference  : [-2388. -2220. -2233. -2199. -2239.]


### What happens if we skip that care

In [62]:
leaky = -cross_val_score(LGBMRegressor(verbose=-1), X_train_team, y_train,
                         cv=folds, scoring='neg_mean_absolute_error')

print('honest:', with_team.round(0), round(with_team.mean()))
print('leaky :', leaky.round(0), round(leaky.mean()))

honest: [1477. 1654. 1558. 1597. 1693.] 1596
leaky : [1348. 1507. 1369. 1464. 1524.] 1442


Team is a huge improvement, K-fold MAE from 3851 to 1596, better in all 5 rounds.

The leaky one (1442) is not a real score. The medians were built before K-fold split the data, so every round already saw its own wages. Its only here to show the leak (154 euro).

### Testing It with Log

In [63]:
model_team_log = LGBMRegressor(verbose=-1)
model_team_log.fit(X_train_team, y_train_log)
pred_team_log = np.expm1(model_team_log.predict(X_test_team))

print('test MAE:', mean_absolute_error(y_test, pred_team_log))
print('RMSE    :', root_mean_squared_error(y_test, pred_team_log))

test MAE: 1484.7639637084221
RMSE    : 6051.349214922003


In [64]:
with_team_log = []

for fit_pos, check_pos in folds.split(X_train):
    X_fit, X_check = X_train.iloc[fit_pos], X_train.iloc[check_pos]
    y_fit, y_check = y_train.iloc[fit_pos], y_train.iloc[check_pos]
    t_fit, t_check = team_train.iloc[fit_pos], team_train.iloc[check_pos]

    X_fit = X_fit.assign(club_wage=club_wage(t_fit, y_fit, t_fit))
    X_check = X_check.assign(club_wage=club_wage(t_fit, y_fit, t_check))

    m = LGBMRegressor(verbose=-1).fit(X_fit, np.log1p(y_fit))   # train on log
    pred = np.expm1(m.predict(X_check))              # back to euro
    with_team_log.append(mean_absolute_error(y_check, pred))

with_team_log = np.array(with_team_log)

print('euro + team:', with_team.round(0), round(with_team.mean()))
print('log + team :', with_team_log.round(0), round(with_team_log.mean()))
print('difference :', (with_team_log - with_team).round(0))

euro + team: [1477. 1654. 1558. 1597. 1693.] 1596
log + team : [1445. 1634. 1421. 1510. 1624.] 1527
difference : [ -31.  -20. -137.  -87.  -68.]


Log + team : MAE lower (1533 to 1485 on test, 1596 to 1527 on K-fold, all 5 rounds better), RMSE a bit higher (5912 to 6051). We want to predict a typical player's wage, not a club's total wage bill, so we use log + team.

## 12. Final Model and Conclusion

### Scoreboard

In [65]:
final = {
    'baseline (median)': pred_baseline,
    'lightgbm': pred_lightgbm,
    'lightgbm + log': pred_log,
    'lightgbm + team': pred_team,
    'lightgbm + team + log': pred_team_log,
}

rows = []
for name, p in final.items():
    rows.append({
        'model': name,
        'MAE': mean_absolute_error(y_test, p),
        'RMSE': root_mean_squared_error(y_test, p),
        'pct_error': (abs(p - y_test) / y_test).mean(),
    })

pd.DataFrame(rows).set_index('model').round(2)

,MAE,RMSE,pct_error
model,,,
baseline (median),7753.57,22049.46,1.45
lightgbm,3973.60,10438.67,1.37
lightgbm + log,3920.92,10739.18,0.89
lightgbm + team,1532.51,5912.02,0.26
lightgbm + team + log,1484.76,6051.35,0.18


Team is the biggest jump (3974 to 1533). Log alone barely did anything, but on top of Team it goes to 1485.

Final model : lightgbm + team + log, it won all 5 K-fold rounds (1527 vs 1596). Trade off : RMSE a bit higher and top earners guessed a bit lower.

### Which score is the honest one

Test says 1485, K-fold says 1527. The test set was used many times to choose (log, Value, Team), so its not really unseen anymore. I report 1527.

Next time : only touch the test set once at the end.

### Diagnosing the final model

In [66]:
result_final = pd.DataFrame({'actual': y_test, 'predicted': pred_team_log})
result_final['error'] = result_final['predicted'] - result_final['actual']
result_final['abs_error'] = result_final['error'].abs()

w = int(len(result_final) * 0.01)
print('top 1% share of error:', result_final.nlargest(w, 'abs_error')['abs_error'].sum() / result_final['abs_error'].sum())

result_final.groupby(pd.cut(result_final['actual'], bands), observed=True).agg(
    n=('error', 'size'),
    mean_error=('error', 'mean'),
    mae=('abs_error', 'mean'),
)

top 1% share of error: 0.24538436681698583


,n,mean_error,mae
actual,,,
"(0, 3000]",2114,60.993752,306.698275
"(3000, 10000]",858,-82.221660,1059.814908
"(10000, 50000]",658,-1046.305943,3410.722433
"(50000, 1000000]",119,-8618.655201,14827.255427


In [67]:
names = pd.read_csv('fifa21_cleaned.csv', usecols=['Name', 'Team'])
result_final.join(names).nlargest(10, 'abs_error')

,actual,predicted,error,abs_error,Name,Team
0,560000.0,305789.132958,-254210.867042,254210.867042,L. Messi,FC Barcelona
3,370000.0,233461.429708,-136538.570292,136538.570292,K. De Bruyne,Manchester City
119,140000.0,52022.317578,-87977.682422,87977.682422,R. Jiménez,Wolverhampton Wanderers
150,100000.0,38713.596144,-61286.403856,61286.403856,Nélson Semedo,Wolverhampton Wanderers
135,90000.0,34810.850148,-55189.149852,55189.149852,Rui Patrício,Wolverhampton Wanderers
17,300000.0,245623.147706,-54376.852294,54376.852294,S. Agüero,Manchester City
315,86000.0,34563.541001,-51436.458999,51436.458999,Jonny,Wolverhampton Wanderers
284,100000.0,62875.579817,-37124.420183,37124.420183,Raphinha,Leeds United
1300,60000.0,27669.723319,-32330.276681,32330.276681,Marçal,Wolverhampton Wanderers
30,230000.0,198592.243934,-31407.756066,31407.756066,Bernardo Silva,Manchester City


Compared with phase 8 :

MAE by band : 1507 / 2873 / 7809 / 34518 -> 307 / 1060 / 3411 / 14827

mean_error : +1172 / +660 / -2186 / -18188 -> +61 / -82 / -1046 / -8619

The sign flip is much smaller. The top 1% share went up (17.2% to 24.5%) only because the total error got smaller.

Still badly wrong : Messi, De Bruyne and 5 Wolverhampton players. Most of the worst clubs are Premier League, so league is probably the next missing info.

### Conclusion

Baseline 7754 -> lightgbm 3974 -> + team 1533 -> + log 1485 (1527 on K-fold). 6 algorithms all stopped around 4000, so the algorithm was never the problem, the missing club info was.

What I got wrong : phase 8 blamed the skewed target. Log only fixed a symptom, Team fixed the cause.

Limitations :
- clubs not in the data just get the overall median
- needs Value, so it cant predict a player with no market value yet
- top earners are guessed too low
- its FIFA 21 data, so it learns EA's wages, not real contracts

Next : a correction for the log pull down, and a league feature.